# Round 3: pilot fine-tune of Qwen2.5-Coder 1.5B and 3B

Rounds 1 and 2 (see `docs/model_selection.md`) left two candidates:

| Model | Base accuracy | Seconds per answer (laptop CPU) |
|---|---|---|
| Qwen2.5-Coder-3B-Instruct | 51.6% | 2.9 |
| Qwen2.5-Coder-1.5B-Instruct | 43.6% | 1.6 |

This notebook fine-tunes both models with identical data and settings, then
exports each one as a 4-bit GGUF file. Evaluation is not done here: the GGUF
files are tested on the laptop with the same sandbox, prompts and runtime as
rounds 1 and 2 (`benchmarks/model_selection/round3_eval.py`), so base and
fine-tuned scores are directly comparable.

**Decision rule (fixed before training, 2026-11-06).** After fine-tuning, the
3B model is kept only if it beats the 1.5B model by at least 5 points of
execution accuracy *and* the difference is significant (paired McNemar test,
p < 0.05). Otherwise the app ships the 1.5B model alone.

**Runtime.** Colab or Kaggle with a T4 GPU. Roughly 15 minutes of training for
1.5B, 30 minutes for 3B, plus about 10 minutes per GGUF export.

## 1. Setup

In [1]:
%%capture
!pip install unsloth

In [2]:
# Unsloth must be imported before trl and transformers so it can patch them.
import unsloth

import gc
import json
import random
import re
import subprocess
import time
from pathlib import Path

import pandas as pd
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download

print(torch.cuda.get_device_name(0))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Tesla T4


Outputs are written to Google Drive when running on Colab, so a finished model
survives a disconnected session. On Kaggle they go to `/kaggle/working`.

In [3]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/ghost-in-the-shell/round3")
except ImportError:
    OUTPUT_DIR = Path("/kaggle/working/round3")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUTPUT_DIR)

Mounted at /content/drive
/content/drive/MyDrive/ghost-in-the-shell/round3


## 2. Configuration

Both models get the same data, seed and hyperparameters. The system prompt is
the one used in rounds 1 and 2, so the fine-tuned models are trained on the
exact format they are tested with.

In [4]:
MODELS = {
    "qwen2.5-coder-1.5b": "unsloth/Qwen2.5-Coder-1.5B-Instruct",
    "qwen2.5-coder-3b": "unsloth/Qwen2.5-Coder-3B-Instruct",
}

SYSTEM_PROMPT = (
    "You translate natural-language requests into a single Linux bash command. "
    "Reply with only the command on one line: no explanation, no markdown, no code fences."
)

SEED = 42
TRAIN_SIZE = 10_000      # pilot run; the full run uses the whole cleaned set
VAL_SIZE = 500
MAX_SEQ_LENGTH = 512

LORA = dict(r=16, lora_alpha=16, lora_dropout=0.0)
BATCH_SIZE, GRAD_ACCUM = 8, 2         # effective batch size 16
TRAINING = dict(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_steps=round(0.03 * TRAIN_SIZE / (BATCH_SIZE * GRAD_ACCUM)),   # 3% of the steps
    weight_decay=0.01,
    optim="adamw_8bit",
)

## 3. Training data

Source: [NL2SH-ALFA](https://huggingface.co/datasets/westenfelder/NL2SH-ALFA)
(MIT license). Its 40,639 training pairs are not manually verified, so they are
cleaned before use:

1. **Decontamination.** Remove any pair whose request or command matches a
   test item after normalisation, and any request whose word overlap with a test
   request is 80% or higher (Jaccard). This keeps the 300-prompt test set unseen.
2. **Quality filters.** Drop duplicates, multi-line commands, commands longer
   than 200 characters, and commands that `bash -n` rejects as invalid syntax.
3. **Sampling.** Draw a fixed random sample for training and validation.

In [5]:
def load_split(name):
    path = hf_hub_download("westenfelder/NL2SH-ALFA", f"{name}.csv", repo_type="dataset")
    return pd.read_csv(path)

train_raw = load_split("train")
test = load_split("test")
print(f"train: {len(train_raw):,}  test: {len(test)}")

train.csv:   0%|          | 0.00/4.91M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/49.5k [00:00<?, ?B/s]

train: 40,639  test: 300


In [6]:
def normalise(text):
    return re.sub(r"\W+", " ", str(text).lower()).strip()

def words(text):
    return set(normalise(text).split())

def jaccard(a, b):
    return len(a & b) / len(a | b) if a | b else 0.0

test_requests = set(test.nl.map(normalise))
test_commands = set(pd.concat([test.bash, test.bash2]).map(normalise))
test_word_sets = [words(t) for t in test.nl]

exact_match = train_raw.nl.map(normalise).isin(test_requests) | train_raw.bash.map(normalise).isin(test_commands)
near_match = train_raw.nl.map(lambda t: max(jaccard(words(t), w) for w in test_word_sets) >= 0.8)

clean = train_raw[~exact_match & ~near_match]
print(f"removed {exact_match.sum()} exact and {(near_match & ~exact_match).sum()} near duplicates of test items")

removed 91 exact and 1 near duplicates of test items


In [7]:
def valid_bash(command):
    return subprocess.run(["bash", "-n", "-c", command], capture_output=True).returncode == 0

clean = clean.dropna().drop_duplicates(subset="nl")
clean = clean[~clean.bash.str.contains("\n") & (clean.bash.str.len() <= 200)]
clean = clean[clean.bash.map(valid_bash)]
print(f"after quality filters: {len(clean):,} pairs")

after quality filters: 38,713 pairs


In [8]:
sample = clean.sample(TRAIN_SIZE + VAL_SIZE, random_state=SEED)
train_df, val_df = sample.iloc[:TRAIN_SIZE], sample.iloc[TRAIN_SIZE:]

def to_messages(df):
    return [
        {"messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row.nl},
            {"role": "assistant", "content": row.bash},
        ]}
        for row in df.itertuples()
    ]

train_records, val_records = to_messages(train_df), to_messages(val_df)

for name, df in [("train", train_df), ("val", val_df)]:
    df.to_csv(OUTPUT_DIR / f"{name}_pilot.csv", index=False)
print(f"train {len(train_records):,}  val {len(val_records)}")
print(train_records[0]["messages"][1:])

train 10,000  val 500
[{'role': 'user', 'content': 'Build a recursive fls list over a device, output paths will start with C'}, {'role': 'assistant', 'content': 'fls -r -m C: /dev/loop1p1'}]


## 4. Fine-tuning

QLoRA with [Unsloth](https://github.com/unslothai/unsloth): the base model is
loaded in 4-bit and low-rank adapters are trained on every attention and MLP
projection. The loss is computed on the assistant's reply only, so the model
learns to produce commands rather than to repeat the prompt.

After training, the adapter is saved, a few test prompts are run as a sanity
check, and the merged model is exported to GGUF `Q4_K_M` (the format the app
runs on the laptop).

In [9]:
import inspect

from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from trl import SFTConfig, SFTTrainer

# TRL renamed two arguments across releases; use whichever this install accepts.
_config_args = inspect.signature(SFTConfig).parameters
SEQ_ARG = "max_length" if "max_length" in _config_args else "max_seq_length"
_trainer_args = inspect.signature(SFTTrainer.__init__).parameters
TOKENIZER_ARG = "processing_class" if "processing_class" in _trainer_args else "tokenizer"


def fine_tune(name, base_model):
    run_dir = OUTPUT_DIR / name
    run_dir.mkdir(exist_ok=True)

    # The T4 has no bfloat16 support, so the model must be loaded in float16 to
    # match the float16 training below.
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=base_model,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None if is_bfloat16_supported() else torch.float16,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        **LORA,
    )

    def render(records):
        texts = [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in records]
        return Dataset.from_dict({"text": texts})

    def build_trainer(mixed_precision):
        trainer = SFTTrainer(
            model=model,
            train_dataset=render(train_records),
            eval_dataset=render(val_records),
            args=SFTConfig(
                dataset_text_field="text",
                output_dir=str(run_dir / "checkpoints"),
                eval_strategy="steps",
                eval_steps=100,
                logging_steps=25,
                save_strategy="no",
                seed=SEED,
                fp16=mixed_precision and not is_bfloat16_supported(),
                bf16=mixed_precision and is_bfloat16_supported(),
                report_to="none",
                **{SEQ_ARG: MAX_SEQ_LENGTH},
                **TRAINING,
            ),
            **{TOKENIZER_ARG: tokenizer},
        )
        return train_on_responses_only(
            trainer,
            instruction_part="<|im_start|>user\n",
            response_part="<|im_start|>assistant\n",
        )

    # If float16 mixed precision still fails on bfloat16 gradients, it fails on
    # the first optimizer step, before any weight is updated. In that case the
    # trainer is rebuilt without mixed precision and training starts again.
    precision = "fp16 mixed precision"
    trainer = build_trainer(mixed_precision=True)
    start = time.time()
    try:
        stats = trainer.train()
    except NotImplementedError as error:
        if "BFloat16" not in str(error):
            raise
        print("float16 mixed precision failed on bfloat16 gradients; retrying without it")
        precision = "no mixed precision (fallback)"
        trainer = build_trainer(mixed_precision=False)
        start = time.time()
        stats = trainer.train()
    minutes = (time.time() - start) / 60
    eval_loss = trainer.evaluate()["eval_loss"]

    model.save_pretrained(run_dir / "lora_adapter")
    tokenizer.save_pretrained(run_dir / "lora_adapter")
    pd.DataFrame(trainer.state.log_history).to_csv(run_dir / "log_history.csv", index=False)

    return model, tokenizer, {
        "model": name,
        "base_model": base_model,
        "train_examples": len(train_records),
        "train_minutes": round(minutes, 1),
        "train_loss": round(stats.training_loss, 4),
        "eval_loss": round(eval_loss, 4),
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "gpu": torch.cuda.get_device_name(0),
        "precision": precision,
    }

In [10]:
def sanity_check(model, tokenizer, n=8):
    FastLanguageModel.for_inference(model)
    rows = test.sample(n, random_state=SEED)
    for row in rows.itertuples():
        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": row.nl}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
        ).to("cuda")
        output = model.generate(**inputs, max_new_tokens=64, do_sample=False)
        prompt_length = inputs["input_ids"].shape[1]
        reply = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True).strip()
        print(f"{row.nl}\n  model:     {reply}\n  reference: {row.bash}\n")


def export_gguf(model, tokenizer, name):
    gguf_dir = OUTPUT_DIR / name / "gguf"
    model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method="q4_k_m")
    # Unsloth writes the file to a sibling folder named "<gguf_dir>_gguf".
    exported = next((OUTPUT_DIR / name).rglob("*Q4_K_M.gguf"))
    target = OUTPUT_DIR / f"{name}-ft.Q4_K_M.gguf"
    exported.rename(target)
    return target


def run(name):
    model, tokenizer, info = fine_tune(name, MODELS[name])
    print(json.dumps(info, indent=2))
    sanity_check(model, tokenizer)
    info["gguf"] = str(export_gguf(model, tokenizer, name))
    (OUTPUT_DIR / name / "run_info.json").write_text(json.dumps(info, indent=2))

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return info

### 4a. Qwen2.5-Coder-1.5B

In [ ]:
info_15b = run("qwen2.5-coder-1.5b")

### 4b. Qwen2.5-Coder-3B

If the session ran out of memory or disconnected after 4a, restart the runtime,
run sections 1 to 3 again, and continue from here. The 1.5B results are already
saved to the output folder.

In [11]:
info_3b = run("qwen2.5-coder-3b")

==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.11.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

... (22 outputs left out)

Search for the directory "mysql" in the workspace directory.
  model:     find $WORKSPACE -name mysql
  reference: find  /workspace -name mysql -type d



Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Dump "/system/text1.txt" as ASCII characters
  model:     cat /system/text1.txt | od -c
  reference: od -t c /system/text1.txt

Calculate the md5 sum of the contents of the sorted list of files "$FILES"
  model:     md5sum $(sort -R $FILES)
  reference: cat $(echo $FILES | tr ' ' '\n' | sort) | md5sum



RuntimeError: Unsloth: Not enough disk space to convert to GGUF.
The export needs about 13.6GB on the filesystem holding `/content/drive/MyDrive/ghost-in-the-shell/round3/qwen2.5-coder-3b/gguf`, which has 1.1GB free.
It writes a 16-bit merge, then a `f16` GGUF, then q4_k_m quants, and the merge and the intermediate are both still on disk while the quants are written.
Options: free space, export fewer quantization methods, point `save_directory` at a bigger filesystem, or push straight to Hugging Face with `.push_to_hub_gguf(...)`.
To skip this check set the environment variable UNSLOTH_DISK_PREFLIGHT=0.

In [12]:
import shutil
from unsloth import FastLanguageModel, is_bfloat16_supported

LOCAL_EXPORT = Path("/content/gguf_export")

# Delete the temporary files the 1.5B export left on Drive. Its final GGUF is kept.
for leftover in ["gguf", "gguf_gguf"]:
    shutil.rmtree(OUTPUT_DIR / "qwen2.5-coder-1.5b" / leftover, ignore_errors=True)
print(f"Drive free: {shutil.disk_usage(OUTPUT_DIR).free / 1e9:.1f} GB")

def export_gguf(model, tokenizer, name):
    work_dir = LOCAL_EXPORT / name
    target = OUTPUT_DIR / f"{name}-ft.Q4_K_M.gguf"
    exported = next(work_dir.rglob("*Q4_K_M.gguf"), None)
    if exported is None:
        model.save_pretrained_gguf(str(work_dir / "gguf"), tokenizer, quantization_method="q4_k_m")
        exported = next(work_dir.rglob("*Q4_K_M.gguf"))
    size = exported.stat().st_size
    free = shutil.disk_usage(OUTPUT_DIR).free
    if free < size:
        raise RuntimeError(f"Drive has {free / 1e9:.1f} GB free, GGUF needs {size / 1e9:.1f} GB. "
                           f"Empty the Drive trash and run this cell again.")
    shutil.copy(exported, target)
    shutil.rmtree(work_dir)
    return target

name = "qwen2.5-coder-3b"
info_3b = {
    "model": name, "base_model": "unsloth/Qwen2.5-Coder-3B-Instruct",
    "train_examples": 10000, "train_minutes": 26.7, "train_loss": 0.98,
    "eval_loss": 0.9578, "trainable_params": 29933568,
    "gpu": "Tesla T4", "precision": "fp16 mixed precision",
}
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(OUTPUT_DIR / name / "lora_adapter"),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None if is_bfloat16_supported() else torch.float16,
)
info_3b["gguf"] = str(export_gguf(model, tokenizer, name))
(OUTPUT_DIR / name / "run_info.json").write_text(json.dumps(info_3b, indent=2))
info_3b

Drive free: 1.1 GB
==((====))==  Unsloth 2026.11.2: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/gguf_export/qwen2.5-coder-3b/gguf/tokenizer_config.json.


... (11 outputs left out)

Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:01<00:00, 60.69s/it]


Unsloth: Merge process complete. Saved to `/content/gguf_export/qwen2.5-coder-3b/gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12128-mix-2f23ac9 (app-b12128-mix-2f23ac9-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/gguf_export/qwen2.5-coder-3b/gguf_gguf/Qwen2.5-Coder-3B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files 

RuntimeError: Drive has 1.1 GB free, GGUF needs 1.9 GB. Empty the Drive trash and run this cell again.

In [13]:
exported = next((LOCAL_EXPORT / name).rglob("*Q4_K_M.gguf"))
final = exported.rename(LOCAL_EXPORT / f"{name}-ft.Q4_K_M.gguf")
info_3b["gguf"] = str(final)
(OUTPUT_DIR / name / "run_info.json").write_text(json.dumps(info_3b, indent=2))
print(final, f"{final.stat().st_size / 1e9:.2f} GB")
!du -sh /content/drive/MyDrive/ghost-in-the-shell/round3/* 2>/dev/null

/content/gguf_export/qwen2.5-coder-3b-ft.Q4_K_M.gguf 1.93 GB
82M	/content/drive/MyDrive/ghost-in-the-shell/round3/qwen2.5-coder-1.5b
941M	/content/drive/MyDrive/ghost-in-the-shell/round3/qwen2.5-coder-1.5b-ft.Q4_K_M.gguf
126M	/content/drive/MyDrive/ghost-in-the-shell/round3/qwen2.5-coder-3b
1.2M	/content/drive/MyDrive/ghost-in-the-shell/round3/train_pilot.csv
63K	/content/drive/MyDrive/ghost-in-the-shell/round3/val_pilot.csv


## 5. Summary

In [ ]:
infos = [json.loads(p.read_text()) for p in sorted(OUTPUT_DIR.glob("*/run_info.json"))]
pd.DataFrame(infos).drop(columns=["gguf"])

In [ ]:
import importlib.metadata

versions = {pkg: importlib.metadata.version(pkg) for pkg in ["unsloth", "transformers", "trl", "peft", "torch"]}
(OUTPUT_DIR / "versions.json").write_text(json.dumps(versions, indent=2))
versions

## 6. Next step: evaluate on the laptop

Copy the two GGUF files from the output folder into `models/` in the project:

```
models/qwen2.5-coder-1.5b-ft.Q4_K_M.gguf
models/qwen2.5-coder-3b-ft.Q4_K_M.gguf
```

Then run, from `benchmarks/model_selection/`:

```
uv run python round3_eval.py
```

It runs the same 300 test prompts, 50 risky prompts and 60 explain-mode
commands as round 2, compares each fine-tuned model with its base model, and
applies the decision rule stated at the top of this notebook.